## Import necessary libraries and functions

In [27]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import functions


# Article-Based Sentiment Model

## Data Set-Up

In [ ]:
# Import dataframes

df_train = pd.read_csv('https://raw.githubusercontent.com/anjalinambudiri-jpg/NFL-Sentiment-Analysis/refs/heads/Web-scraping-Function/Training_Articles_withText.csv')
df_test = pd.read_csv('https://raw.githubusercontent.com/anjalinambudiri-jpg/NFL-Sentiment-Analysis/refs/heads/Web-scraping-Function/Test_Articles_with_Text.csv')


In [ ]:
# Encode relevant categorical variables for both dataframes

df_test = functions.label(df_test, ['Quarterback', 'Article Subset'])
df_train = functions.label(df_train, ['Sentiment'])

df_train.head()

,Date,Topic/Name,Source,Link,Sentiment,Text
0,6/14/2023,Fines for Hold Outs,NBCNY,https://www.nbcnewyork.com/news/sports/nfl/do-...,1,It's holdout season. NFL teams are stepping ba...
1,8/10/2023,One-year contracts that look promising,NFL,https://www.nfl.com/news/2023-nfl-season-ten-o...,2,Every player wants the security of a long-term...
2,7/25/2023,2023 off-season updates,ESPN,https://www.espn.com/nfl/story/_/id/38042234/2...,1,The NFL offseason is always long and eventful....
3,6/13/2023,NFL holdouts,YS,https://sports.yahoo.com/nfl-holdouts-chris-jo...,3,"Every year, NFL stars skip mandatory team prac..."
4,6/1/2023,Importance of practice in off-season,FS,https://www.foxsports.com/stories/nfl/why-prac...,4,"""We're talking about practice!""Somewhere there..."


In [ ]:

# Lemmatize the text in training and test dataframes (lowercase, same words of different tenses collapsed)

df_train['Cleaned_Text'] = df_train['Text'].apply(functions.lemmatize)
df_test['Cleaned_Text'] = df_test['Text'].apply(functions.lemmatize)

df_test



,Quarterback,Team,Article Subset,Date,Topic/Name,Source,Link,Text,Cleaned_Text
0,5,Detroit Lions,1,4/26/2025,Draft Picks,ESPN,https://www.espn.com/nfl/story/_/id/44566938/d...,The 2025 NFL draft began Thursday in Green Bay...,the 2025 nfl draft began thursday in green bay...
1,5,Detroit Lions,1,6/10/2025,Thoughts on offence,NYT,https://www.nytimes.com/athletic/6414883/2025/...,DETROIT — One thought for every offensive play...,detroit — one thought for every offensive play...
2,5,Detroit Lions,1,7/17/2025,Offseason updates,NFL,https://www.nfl.com/news/detroit-lions-trainin...,"With 2025 NFL training camps set to open, it's...","with 2025 nfl training camp set to open , it '..."
3,5,Detroit Lions,1,7/17/2025,New OC,NFL,https://www.nfl.com/news/lions-te-sam-laporta-...,With Detroit Lions veterans reporting for trai...,with detroit lion veteran reporting for traini...
4,5,Detroit Lions,1,6/2/2025,Jared Goff thoughts on new coach,NFL,https://www.nfl.com/news/jared-goff-not-fretti...,The Detroit Lions' No. 1 scoring offense lost ...,the detroit lion ' no . 1 scoring offense lost...
...,...,...,...,...,...,...,...,...,...
145,1,Carolina Panthers,0,11/16/2025,"Bryce Young overcomes ankle injury, sets Panth...",ESPN,https://www.espn.com/nfl/story/_/id/46993193/b...,ATLANTA -- Carolina Panthers quarterback Bryce...,atlanta -- carolina panther quarterback bryce ...
146,1,Carolina Panthers,0,11/25/2025,Bryce Young after Panthers' loss: 'It's no one...,Al,https://www.al.com/sports/2025/11/bryce-young-...,The Carolina Panthers came into their first pr...,the carolina panther came into their first pri...
147,1,Carolina Panthers,0,11/21/2025,NFL analyst 'overwhelmingly impressed' by Pant...,SI,https://www.si.com/nfl/panthers/onsi/news/nfl-...,It's hard to understate just how good Bryce Yo...,it 's hard to understate just how good bryce y...
148,1,Carolina Panthers,0,12/26/2025,Bryce Young powers up 4 more spots in NFL QB p...,SI,https://www.si.com/nfl/panthers/onsi/bryce-you...,"Earlier this week, Garrett Podell of CBS Sport...","earlier this week , garrett podell of cbs spor..."


In [ ]:
# Collapse sentiment labels from 5 categories to 3 to improve accuracy

mapping = {
    2: 2, 4:2,  # Let 'slightly positive' and 'positive' represent the same category
    1: 1,
    3: 0, 0:0  # Let 'slightly negative' and 'negative' represent the same category
}



In [ ]:
# Define Predictor, Target for Training Dataframe
X = df_train['Cleaned_Text']
Y = df_train['Sentiment'].map(mapping)

## Testing Pipelines

In [ ]:
# Import all libraries
 
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression



In [ ]:
# Create initial pipeline, using Logistic Regression

LR = Pipeline([
        ('vect', CountVectorizer()),
        ('tfidf', TfidfTransformer()),
        ('logr', LogisticRegression(class_weight='balanced',max_iter=1000))]) #Less negative articles in dataset, so we choose 'balanced' class_weight

In [ ]:
# Train and compute accuracy
print(functions.train(LR, X, Y))

Logistic Regression Accuracy Trial 1: 0.6176470588235294
Logistic Regression Accuracy Trial 2: 0.5294117647058824
Logistic Regression Accuracy Trial 3: 0.5588235294117647
Logistic Regression Accuracy Trial 4: 0.5
Logistic Regression Accuracy Trial 5: 0.4117647058823529
0.5235294117647059


In [ ]:
# Create new pipeline: Logistic Regression, negative articles weighted more heavily

custom_weights = {
        0: 3.5,  # Negative: Heavily penalized if missed
        1: 1.0,  # Neutral
        2: 1.5   # Positive: Slightly boosted
    }

LR1 = Pipeline([
        ('vect', CountVectorizer()), 
        ('tfidf', TfidfTransformer()),
        ('logr', LogisticRegression(class_weight=custom_weights,max_iter=1000))])

In [ ]:
#Train and compute accuracy
print(functions.train(LR1, X, Y))

Logistic Regression Accuracy Trial 1: 0.5
Logistic Regression Accuracy Trial 2: 0.4411764705882353
Logistic Regression Accuracy Trial 3: 0.5
Logistic Regression Accuracy Trial 4: 0.4117647058823529
Logistic Regression Accuracy Trial 5: 0.4117647058823529
0.45294117647058824


In [ ]:
from sklearn.naive_bayes import ComplementNB
# Create new pipeline using ComplementNB
CNB = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer(sublinear_tf=True)),
    ('cnb', ComplementNB()) 
])

In [56]:
functions.train(CNB, X, Y)


Logistic Regression Accuracy Trial 1: 0.3235294117647059
Logistic Regression Accuracy Trial 2: 0.5588235294117647
Logistic Regression Accuracy Trial 3: 0.5
Logistic Regression Accuracy Trial 4: 0.6176470588235294
Logistic Regression Accuracy Trial 5: 0.47058823529411764


np.float64(0.49411764705882355)

## All pipelines perform somewhat similarly:
- Logistic Regression, balanced weights: 52%
- Logistic Regression, custom weights: 45%
- ComplementNB: 49%

### Since its accuracy was the highest, we will proceed with Logistic Regression, balanced weights.


## Tune Hyperparameters using GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Create Basic pipeline
optimization_pipeline = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('logr', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42))
])

# Define hyperparamters that we will tune
param_grid = {
    
    'tfidf__sublinear_tf': [True, False], # Account for words used frequently, but in long articles 
   
    'vect__ngram_range': [(1, 1), (1, 2)], # Use Unigrams (single words) or Unigrams and Bigrams (word pairs)
    
    'vect__max_features': [200, 400, 600, 800, 1000, None], # Restrict vocabulary size
    
    
    'vect__min_df': [1, 2, 3], # Exclude words that don't appear in at least X different articles
    
    
    'logr__C': [0.01, 0.1, 1.0, 5.0, 10.0] # Tune regularization strength 
}


cv = StratifiedKFold() # We choose StratifiedKFold since there are multiple categories for samples to be classified into (Negative, Neutral, Positive)

# Initialize Grid Search
grid_search = GridSearchCV(
    estimator=optimization_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy', # Recommended for classification
    n_jobs=-1 # Improve efficiency
)

# Run on Predictor, Target columns
grid_search.fit(X, Y)

# Output results

print("Best Hyperparameter Configuration:")
for param, val in grid_search.best_params_.items():
    print(f"  {param}: {val}")

# Save pipeline with the best parameters
finetuned_model = grid_search.best_estimator_

Best Hyperparameter Configuration:
  logr__C: 10.0
  tfidf__sublinear_tf: True
  vect__max_features: None
  vect__min_df: 1
  vect__ngram_range: (1, 1)


In [ ]:
#Assess accuracy of finetuned_model
functions.train(finetuned_model, X, Y)

Logistic Regression Accuracy Trial 1: 0.5882352941176471
Logistic Regression Accuracy Trial 2: 0.5588235294117647
Logistic Regression Accuracy Trial 3: 0.6176470588235294
Logistic Regression Accuracy Trial 4: 0.6764705882352942
Logistic Regression Accuracy Trial 5: 0.5588235294117647


np.float64(0.6)

## Create sentiment probabilities for unlabeled articles in Test Dataframe

In [41]:
# Define predictor for the test dataframe
X_test = df_test['Cleaned_Text']

# Use the finetuned model to predict sentiment probabilities on the test set
test_probabilities = finetuned_model.predict_proba(X_test)

# Create a weighted sentiment score for each article
sentiment_weights = np.array([-1, 0, 1])
df_test['Sentiment_Score'] = np.dot(test_probabilities, sentiment_weights)

# Save the raw probabilities
df_test['Prob_Negative'] = test_probabilities[:, 0]
df_test['Prob_Neutral'] = test_probabilities[:, 1]
df_test['Prob_Positive'] = test_probabilities[:, 2]



In [42]:
df_test

,Quarterback,Team,Article Subset,Date,Topic/Name,Source,Link,Text,Cleaned_Text,Sentiment_Score,Prob_Negative,Prob_Neutral,Prob_Positive
0,5,Detroit Lions,1,4/26/2025,Draft Picks,ESPN,https://www.espn.com/nfl/story/_/id/44566938/d...,The 2025 NFL draft began Thursday in Green Bay...,the 2025 nfl draft began thursday in green bay...,0.329933,0.204372,0.261323,0.534305
1,5,Detroit Lions,1,6/10/2025,Thoughts on offence,NYT,https://www.nytimes.com/athletic/6414883/2025/...,DETROIT — One thought for every offensive play...,detroit — one thought for every offensive play...,0.396546,0.104268,0.394918,0.500814
2,5,Detroit Lions,1,7/17/2025,Offseason updates,NFL,https://www.nfl.com/news/detroit-lions-trainin...,"With 2025 NFL training camps set to open, it's...","with 2025 nfl training camp set to open , it '...",0.241184,0.161028,0.436760,0.402212
3,5,Detroit Lions,1,7/17/2025,New OC,NFL,https://www.nfl.com/news/lions-te-sam-laporta-...,With Detroit Lions veterans reporting for trai...,with detroit lion veteran reporting for traini...,0.304665,0.217617,0.260102,0.522281
4,5,Detroit Lions,1,6/2/2025,Jared Goff thoughts on new coach,NFL,https://www.nfl.com/news/jared-goff-not-fretti...,The Detroit Lions' No. 1 scoring offense lost ...,the detroit lion ' no . 1 scoring offense lost...,0.449983,0.173402,0.203212,0.623385
...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,1,Carolina Panthers,0,11/16/2025,"Bryce Young overcomes ankle injury, sets Panth...",ESPN,https://www.espn.com/nfl/story/_/id/46993193/b...,ATLANTA -- Carolina Panthers quarterback Bryce...,atlanta -- carolina panther quarterback bryce ...,0.250910,0.245948,0.257194,0.496858
146,1,Carolina Panthers,0,11/25/2025,Bryce Young after Panthers' loss: 'It's no one...,Al,https://www.al.com/sports/2025/11/bryce-young-...,The Carolina Panthers came into their first pr...,the carolina panther came into their first pri...,0.243385,0.287053,0.182509,0.530438
147,1,Carolina Panthers,0,11/21/2025,NFL analyst 'overwhelmingly impressed' by Pant...,SI,https://www.si.com/nfl/panthers/onsi/news/nfl-...,It's hard to understate just how good Bryce Yo...,it 's hard to understate just how good bryce y...,0.345264,0.232692,0.189351,0.577956
148,1,Carolina Panthers,0,12/26/2025,Bryce Young powers up 4 more spots in NFL QB p...,SI,https://www.si.com/nfl/panthers/onsi/bryce-you...,"Earlier this week, Garrett Podell of CBS Sport...","earlier this week , garrett podell of cbs spor...",0.352251,0.234090,0.179568,0.586341


In [43]:
#Group the articles by their player and type, and calculate a mean sentiment score for each
df_results = df_test.groupby(["Quarterback", 'Article Subset']).mean("Sentiment_Score")


# Add a column with the overall average sentiment score for each quarterback amongst all articles
df_results['Overall_QB_Average'] = (
    df_results.groupby("Quarterback")["Sentiment_Score"]
    .transform("mean")
)

df_results

Sentiment_Score  Prob_Negative  Prob_Neutral  \
Quarterback Article Subset                                                 
0           0                      0.356108       0.228863      0.186166   
            1                      0.100487       0.295292      0.308929   
            2                      0.139233       0.312261      0.236245   
1           0                      0.276366       0.254496      0.214643   
            1                      0.163726       0.290739      0.254797   
            2                      0.067309       0.338942      0.254806   
2           0                      0.025620       0.416461      0.141457   
            1                      0.217719       0.236684      0.308913   
            2                      0.241398       0.295248      0.168107   
3           0                      0.095094       0.296551      0.311805   
            1                      0.257169       0.239154      0.264523   
            2                      0.220110       0.298659      0.182571   
4           0                      0.423909       0.216342      0.143408   
            1                      0.363172       0.232938      0.170951   
            2                      0.567466       0.147093      0.138348   
5           0                      0.436252       0.205559      0.152630   
            1                      0.344462       0.172137      0.311263   
            2                      0.416114       0.177013      0.229859   
6           0                      0.315421       0.268279      0.148020   
            1                      0.334155       0.221937      0.221970   
            2                      0.343001       0.234554      0.187891   
7           0                     -0.007019       0.360430      0.286159   
            1                      0.214719       0.290969      0.203343   
            2                      0.289899       0.244571      0.220959   
8           0                      0.167443       0.252762      0.327034   
            1                     -0.037001       0.387447      0.262108   
            2                     -0.013875       0.349308      0.315259   
9           0                      0.023853       0.381028      0.214091   
            1                      0.309389       0.185760      0.319091   
            2                      0.206578       0.252141      0.289140   

                            Prob_Positive  Overall_QB_Average  
Quarterback Article Subset                                     
0           0                    0.584971            0.198610  
            1                    0.395779            0.198610  
            2                    0.451494            0.198610  
1           0                    0.530861            0.169134  
            1                    0.454465            0.169134  
            2                    0.406251            0.169134  
2           0                    0.442082            0.161579  
            1                    0.454403            0.161579  
            2                    0.536645            0.161579  
3           0                    0.391644            0.190791  
            1                    0.496323            0.190791  
            2                    0.518770            0.190791  
4           0                    0.640250            0.451516  
            1                    0.596111            0.451516  
            2                    0.714559            0.451516  
5           0                    0.641811            0.398943  
            1                    0.516600            0.398943  
            2                    0.593128            0.398943  
6           0                    0.583701            0.330859  
            1                    0.556092            0.330859  
            2                    0.577555            0.330859  
7           0                    0.353411            0.165866  
            1                    0.505688            0.1658

In [44]:
names = {
    0: 'Bo Nix',
    1: 'Bryce Young',
    2: 'CJ Stroud',
    3: 'Geno Smith',
    4: 'Jalen Hurts',
    5: 'Jared Goff',
    6: 'Justin Herbert',
    7: 'Lamar Jackson',
    8: 'Matthew Stafford',
    9: 'Patrick Mahomes'
}

article_type = {
    0: 'In-Season', 
    1: 'Offense', 
    2: 'Player'
}

In [45]:
# Add names, article types back in
df_results['QB Name'] = df_results.index.get_level_values('Quarterback').map(names)
df_results['Subset Label'] = df_results.index.get_level_values('Article Subset').map(article_type)

df_results

Sentiment_Score  Prob_Negative  Prob_Neutral  \
Quarterback Article Subset                                                 
0           0                      0.356108       0.228863      0.186166   
            1                      0.100487       0.295292      0.308929   
            2                      0.139233       0.312261      0.236245   
1           0                      0.276366       0.254496      0.214643   
            1                      0.163726       0.290739      0.254797   
            2                      0.067309       0.338942      0.254806   
2           0                      0.025620       0.416461      0.141457   
            1                      0.217719       0.236684      0.308913   
            2                      0.241398       0.295248      0.168107   
3           0                      0.095094       0.296551      0.311805   
            1                      0.257169       0.239154      0.264523   
            2                      0.220110       0.298659      0.182571   
4           0                      0.423909       0.216342      0.143408   
            1                      0.363172       0.232938      0.170951   
            2                      0.567466       0.147093      0.138348   
5           0                      0.436252       0.205559      0.152630   
            1                      0.344462       0.172137      0.311263   
            2                      0.416114       0.177013      0.229859   
6           0                      0.315421       0.268279      0.148020   
            1                      0.334155       0.221937      0.221970   
            2                      0.343001       0.234554      0.187891   
7           0                     -0.007019       0.360430      0.286159   
            1                      0.214719       0.290969      0.203343   
            2                      0.289899       0.244571      0.220959   
8           0                      0.167443       0.252762      0.327034   
            1                     -0.037001       0.387447      0.262108   
            2                     -0.013875       0.349308      0.315259   
9           0                      0.023853       0.381028      0.214091   
            1                      0.309389       0.185760      0.319091   
            2                      0.206578       0.252141      0.289140   

                            Prob_Positive  Overall_QB_Average  \
Quarterback Article Subset                                      
0           0                    0.584971            0.198610   
            1                    0.395779            0.198610   
            2                    0.451494            0.198610   
1           0                    0.530861            0.169134   
            1                    0.454465            0.169134   
            2                    0.406251            0.169134   
2           0                    0.442082            0.161579   
            1                    0.454403            0.161579   
            2                    0.536645            0.161579   
3           0                    0.391644            0.190791   
            1                    0.496323            0.190791   
            2                    0.518770            0.190791   
4           0                    0.640250            0.451516   
            1                    0.596111            0.451516   
            2                    0.714559            0.451516   
5           0                    0.641811            0.398943   
            1                    0.516600            0.398943   
            2                    0.593128            0.398943   
6           0                    0.583701            0.330859   
            1                    0.556092            0.330859   
            2                    0.577555            0.330859   
7           0                    0.353411            0.165866   
            1                    0.

In [46]:
#Save results as csv
df_results.to_csv('article_sentiment_model_results', index=False)

# Vader Model

## Import data

In [47]:
# Create dataframe from csv
df_vader = pd.read_csv('https://raw.githubusercontent.com/anjalinambudiri-jpg/NFL-Sentiment-Analysis/refs/heads/Web-scraping-Function/Test_Articles_with_Text.csv')
df_vader.head()

,Quarterback,Team,Article Subset,Date,Topic/Name,Source,Link,Text
0,Jared Goff,Detroit Lions,Offense,4/26/2025,Draft Picks,ESPN,https://www.espn.com/nfl/story/_/id/44566938/d...,The 2025 NFL draft began Thursday in Green Bay...
1,Jared Goff,Detroit Lions,Offense,6/10/2025,Thoughts on offence,NYT,https://www.nytimes.com/athletic/6414883/2025/...,DETROIT — One thought for every offensive play...
2,Jared Goff,Detroit Lions,Offense,7/17/2025,Offseason updates,NFL,https://www.nfl.com/news/detroit-lions-trainin...,"With 2025 NFL training camps set to open, it's..."
3,Jared Goff,Detroit Lions,Offense,7/17/2025,New OC,NFL,https://www.nfl.com/news/lions-te-sam-laporta-...,With Detroit Lions veterans reporting for trai...
4,Jared Goff,Detroit Lions,Offense,6/2/2025,Jared Goff thoughts on new coach,NFL,https://www.nfl.com/news/jared-goff-not-fretti...,The Detroit Lions' No. 1 scoring offense lost ...


## Prepare data

In [ ]:
#Turn categorical variables into numerics
df_vader = functions.label(df_test, ['Quarterback', 'Article Subset'])

In [ ]:
#Calculate sentiment score for each article using find_sentiment_dict
sentiment_df = df_vader['Text'].apply(functions.find_sentiment_dict).apply(pd.Series)
df_vader = pd.concat([df_vader, sentiment_df], axis=1)

In [ ]:
#Group the articles by their player and type, and calculate a mean sentiment score for each
sentiment_averages = df_vader.groupby(['Quarterback', 'Article Subset'])[['neg', 'neu', 'pos', 'compound']].mean()

sentiment_averages

neg     neu     pos  compound
Quarterback Article Subset                                  
0           0               0.0498  0.8252  0.1250   0.99246
            1               0.0354  0.8502  0.1144   0.98042
            2               0.0554  0.8296  0.1150   0.91116
1           0               0.0748  0.8222  0.1028   0.32160
            1               0.0518  0.8510  0.0974   0.92852
            2               0.0728  0.7806  0.1466   0.93490
2           0               0.0856  0.8338  0.0806   0.20828
            1               0.0792  0.8200  0.1008   0.55668
            2               0.0642  0.8222  0.1140   0.44032
3           0               0.0604  0.8064  0.1332   0.97612
            1               0.0474  0.8504  0.1022   0.61656
            2               0.0596  0.8322  0.1078   0.86178
4           0               0.0956  0.8102  0.0942  -0.43012
            1               0.0676  0.8024  0.1298   0.98648
            2               0.0958  0.7792  0.1248  -0.17014
5           0               0.0712  0.8376  0.0910   0.35202
            1               0.0540  0.8320  0.1138   0.62190
            2               0.0616  0.8110  0.1278   0.97572
6           0               0.0486  0.8530  0.0982   0.94574
            1               0.0552  0.8204  0.1242   0.96008
            2               0.0448  0.8396  0.1156   0.92566
7           0               0.0858  0.8012  0.1132   0.45946
            1               0.0432  0.8544  0.1024   0.95818
            2               0.0582  0.8364  0.1054   0.26172
8           0               0.0382  0.8004  0.1616   0.99420
            1               0.0510  0.8492  0.0998   0.34408
            2               0.0332  0.8720  0.0946   0.89532
9           0               0.0684  0.8392  0.0926   0.20856
            1               0.0620  0.7910  0.1472   0.89454
            2               0.0472  0.8418  0.1110   0.98674

In [51]:
# Download dataframe as csv
sentiment_averages.to_csv('vader_sentiment_dataframe', index=False)